In [3]:
import sys
sys.path.append('/home/caoren/tmp/PIRC_for_HigherOrderCausality')
from Model.this import *
import os
os.chdir("/home/caoren/tmp/PIRC_for_HigherOrderCausality/Results/EEG")
import numpy as np
from Torch_Library import *
from PIRC import PIRC_flatten as Nonliear_PIRC
from Reconstruction_PIRC import *
import numpy as np
from itertools import combinations
import argparse
import math
import numpy as np
import matplotlib.pyplot as plt
import pickle
from collections import Counter
import optuna
optuna.logging.set_verbosity(optuna.logging.CRITICAL)

/home/caoren/anaconda3/envs/pytorch/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# =========================
# Tools
# =========================

def build_T_to_Ainf(T_from_Ainf, max_order=3):
    T_to_Ainf = {o: [] for o in range(2, max_order + 1)}

    row, col = T_from_Ainf.shape
    node_num = row

    for i in range(row):
        for j in range(col):
            val = T_from_Ainf[i, j]

            if val == 0:
                continue

            if j < node_num:
                if j != i:
                    T_to_Ainf[2].append([i, j, val])

            elif j < node_num + math.comb(node_num - 1, 2):
                idx = j - node_num
                others = [x for x in range(node_num) if x != i]
                comb_list = list(combinations(others, 2))
                j0, k0 = comb_list[idx]
                T_to_Ainf[3].append([i, j0, k0, val])

            elif max_order == 4:
                idx = j - node_num - math.comb(node_num - 1, 2)
                others = [x for x in range(node_num) if x != i]
                comb_list = list(combinations(others, 3))
                j0, k0, l0 = comb_list[idx]
                T_to_Ainf[4].append([i, j0, k0, l0, val])

    for o in T_to_Ainf:
        T_to_Ainf[o] = (
            np.array(T_to_Ainf[o])
            if len(T_to_Ainf[o]) > 0
            else np.zeros((0, o + 1))
        )

    return T_to_Ainf


def top_freq_per_sequence(Ainf_list, order=3, top_n=5, threshold=None):
    """
    每个序列：
    1. 可选：剔除 |weight| < threshold 的边
    2. 按 |weight| 从大到小排序
    3. 取 top_n
    4. 统计超边出现频率
    """
    all_top_edges = []

    for Ainf in Ainf_list:
        if order not in Ainf or Ainf[order].shape[0] == 0:
            continue

        edges = Ainf[order]
        weights = np.abs(edges[:, -1])

        if threshold is not None:
            valid = weights >= threshold
            edges = edges[valid]
            weights = weights[valid]

        if len(weights) == 0:
            continue

        k = min(top_n, len(weights))
        idx_top = np.argsort(weights)[-k:][::-1]

        top_edges = edges[idx_top, :order]

        for row in top_edges:
            row = row.astype(int)

            if order == 3:
                target = int(row[0])
                j, k0 = sorted([int(row[1]), int(row[2])])
                edge = (target, j, k0)
            else:
                edge = tuple(row)

            all_top_edges.append(edge)

    return Counter(all_top_edges)


def high_order_ratio(Score, n=7):
    if torch.is_tensor(Score):
        Score = Score.detach().cpu().numpy()

    Score = np.asarray(Score).copy()

    for i in range(n):
        Score[i, i] = 0.0

    higher = Score[:, n:].sum()
    total = Score.sum()

    return higher / (total + 1e-12)


def normalize_data(X0, norm):
    if norm == 1:
        scale = np.mean(np.abs(X0))
        if scale < 1e-12:
            return None
        return X0 / scale

    elif norm == 2:
        return (X0 - X0.mean(axis=1, keepdims=True)) / (
            X0.std(axis=1, keepdims=True) + 1e-12
        )

    else:
        raise ValueError("norm must be 1 or 2")

def Causal_score(y_pred, y_true, method="exp", tau=2, eps=1e-12):

    # 转成 tensor
    y_pred = torch.as_tensor(y_pred)
    y_true = torch.as_tensor(y_true)

    # mse over time dimension only
    mse = (y_pred - y_true).pow(2).mean(dim=-1).abs()         # (...,)

    # scale: mean over time of y_true (同你原实现)
    # scale = y_true.mean(dim=-1).abs().clamp_min(eps)          # (...,)
    scale1 = y_true.pow(2).mean(dim=-1).abs()
    scale2 = y_true.var(dim=-1, correction=0)# (...,)

    return torch.sqrt(mse / scale1)
    return torch.sqrt(mse / scale2)

In [6]:
with open("search_target0_results2.pkl", "rb") as f:
    data = pickle.load(f)
device = torch.device("cuda:2")
all_results = data["all_results"]
valid_results = data["valid_results"]

Test_results = valid_results[0]
I_type=Test_results["I_type"]
N_train=Test_results["N_train"]
norm=Test_results["norm"]
Win=Test_results["Win"]
Win=torch.as_tensor(Win, dtype=torch.float32, device=device)
Wres=Test_results["Wres"]
Wres=torch.as_tensor(Wres, dtype=torch.float32, device=device)
params=Test_results["params"]
n_units = params["n_units"]
block_dim = params["block_dim"]
connectivity=params["connectivity"]

args = argparse.Namespace(
        dt=1 / 160,
        n=7,
        N_train=N_train,
        N_test=10,
        N_washout=0,
        N_start=0,
        norm=norm,
        I_type=I_type,
    )